# Module 05-3: YAML 파일에서 프롬프트 로드

## 학습 목표
- 프롬프트를 YAML 파일로 분리하는 이유와 방법을 이해할 수 있다
- `load_prompt()` 함수를 구현하여 YAML에서 `ChatPromptTemplate`을 생성할 수 있다
- Few-shot 예시가 포함된 YAML 프롬프트를 로드하고 사용할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/05-prompt-engineering.md` 섹션 2.3

## 개념 요약

### 왜 프롬프트를 YAML로 분리하는가?

코드에 프롬프트를 하드코딩하면:
- 프롬프트 수정 시마다 코드 배포가 필요
- 변경 이력 추적 어려움
- A/B 테스트 불가능

**YAML 분리의 장점**:
- 코드 재배포 없이 프롬프트 수정 가능
- Git으로 버전 관리
- 환경변수로 버전 동적 전환 가능

### YAML 프롬프트 파일 구조

```yaml
version: "v1"
description: "설명"

system: |
  시스템 메시지 내용

human: |
  {변수}를 포함한 human 메시지

few_shots:  # 선택 사항
  - input: "예시 입력"
    output: "예시 출력"
```

In [ ]:
# 환경 설정
import sys
import os
sys.path.insert(0, '..')

import yaml
from pathlib import Path
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from common.fake_llm import FakeLLM

# 현재 노트북이 있는 디렉토리 확인
NOTEBOOK_DIR = Path(".").resolve()
print(f"현재 디렉토리: {NOTEBOOK_DIR}")
print("환경 설정 완료")

## 실습 1: YAML 파일 내용 확인

미리 준비된 `prompts/code_analyzer.yaml` 파일을 읽고 구조를 이해합니다.

In [ ]:
# 1-1. YAML 파일 경로 확인
yaml_path = Path("prompts/code_analyzer.yaml")

if yaml_path.exists():
    with open(yaml_path, encoding="utf-8") as f:
        content = f.read()
    print("=== YAML 파일 내용 ===")
    print(content)
else:
    print(f"파일을 찾을 수 없습니다: {yaml_path.resolve()}")
    print("현재 디렉토리:", Path(".").resolve())

In [ ]:
# 1-2. yaml.safe_load()로 파싱
with open(yaml_path, encoding="utf-8") as f:
    data = yaml.safe_load(f)

print("파싱된 데이터 구조:")
print(f"  version: {data.get('version')}")
print(f"  description: {data.get('description')}")
print(f"  system 키: {'있음' if 'system' in data else '없음'}")
print(f"  human 키: {'있음' if 'human' in data else '없음'}")
print(f"  few_shots 키: {'있음' if 'few_shots' in data else '없음'}")

In [ ]:
# 검증 셀
assert yaml_path.exists(), f"prompts/code_analyzer.yaml 파일이 존재해야 합니다."
assert "version" in data, "YAML에 version 키가 있어야 합니다."
assert "system" in data, "YAML에 system 키가 있어야 합니다."
print("실습 1 완료!")

## 실습 2: load_prompt() 함수 구현

YAML 파일에서 `ChatPromptTemplate`을 생성하는 `load_prompt()` 함수를 구현합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): YAML을 로드하여 system/few_shots/human 키를 읽고
#               messages 리스트에 순서대로 추가한 뒤 ChatPromptTemplate.from_messages()를 반환합니다.
#
# 힌트 2 (키워드): yaml.safe_load(), messages = [("system", ...), ("human", ...), ("assistant", ...)]
#               few_shots 처리: for example in data.get("few_shots", [])
#
# 힌트 3 (거의 정답):
#   def load_prompt(yaml_path: str | Path) -> ChatPromptTemplate:
#       with open(yaml_path, encoding="utf-8") as f:
#           data = yaml.safe_load(f)
#       messages = [("system", data["system"])]
#       for example in data.get("few_shots", []):
#           messages.append(("human", example["input"]))
#           messages.append(("assistant", example["output"]))
#       messages.append(("human", data["human"]))
#       return ChatPromptTemplate.from_messages(messages)

def load_prompt(yaml_path: str) -> ChatPromptTemplate:
    """YAML 파일에서 ChatPromptTemplate을 로드한다.
    
    Args:
        yaml_path: YAML 파일 경로
    
    Returns:
        ChatPromptTemplate 인스턴스
    """
    # TODO: 구현하세요
    pass

In [ ]:
# 2-2. 함수 테스트
prompt_from_yaml = load_prompt("prompts/code_analyzer.yaml")

if prompt_from_yaml:
    print("YAML에서 로드된 프롬프트:")
    print(f"  메시지 수: {len(prompt_from_yaml.messages)}")
    for i, msg in enumerate(prompt_from_yaml.messages):
        # PromptTemplate의 template 속성 또는 content 속성 사용
        content = getattr(msg.prompt, 'template', str(msg))[:80]
        print(f"  [{i+1}] {msg.__class__.__name__}: {content}...")

In [ ]:
# 검증 셀
assert prompt_from_yaml is not None, "load_prompt()가 None을 반환합니다. 함수를 구현하세요."
assert isinstance(prompt_from_yaml, ChatPromptTemplate), "반환값이 ChatPromptTemplate이어야 합니다."
assert len(prompt_from_yaml.messages) >= 2, "최소 system + human 2개 메시지가 있어야 합니다."
print("실습 2 완료!")

## 실습 3: Few-shot이 포함된 YAML 프롬프트 생성 및 로드

Few-shot 예시가 포함된 새로운 YAML 파일을 생성하고 로드합니다.

In [ ]:
# 3-1. Few-shot 포함 YAML 파일 생성
few_shot_yaml_content = """version: "v1"
description: "이메일 분류 프롬프트 (Few-shot 포함)"
created_at: "2026-03-25"

system: |
  이메일을 분류하세요. 카테고리: 문의, 불만, 칭찬, 기타
  반드시 JSON 형식으로만 응답하세요.

few_shots:
  - input: "이메일 내용: 환불 절차가 어떻게 되나요?"
    output: '{"category": "문의"}'
  - input: "이메일 내용: 배송이 2주나 지연되어 정말 화가 납니다."
    output: '{"category": "불만"}'
  - input: "이메일 내용: 친절한 상담 감사합니다. 문제 해결했어요!"
    output: '{"category": "칭찬"}'

human: |
  이메일 내용: {email}
"""

# prompts 디렉토리에 저장
Path("prompts").mkdir(exist_ok=True)
with open("prompts/email_classifier.yaml", "w", encoding="utf-8") as f:
    f.write(few_shot_yaml_content)

print("prompts/email_classifier.yaml 생성 완료")

In [ ]:
# 3-2. Few-shot YAML 로드 및 테스트
email_prompt = load_prompt("prompts/email_classifier.yaml")

print(f"로드된 메시지 수: {len(email_prompt.messages)}")

# FakeLLM으로 테스트
email_llm = FakeLLM(responses={
    "환불": '{"category": "문의"}',
    "지연": '{"category": "불만"}',
})

email_chain = email_prompt | email_llm | StrOutputParser()
result = email_chain.invoke({"email": "환불 신청을 하고 싶습니다."})
print(f"분류 결과: {result}")

In [ ]:
# 검증 셀
assert email_prompt is not None, "email_prompt가 None입니다."
# system(1) + 3쌍의 few-shot(6) + human(1) = 8
assert len(email_prompt.messages) == 8, f"메시지가 8개이어야 합니다. 현재: {len(email_prompt.messages)}개"
print("실습 3 완료!")

## 정리

이번 실습에서 학습한 내용:

1. **YAML 분리의 이유**: 코드 재배포 없이 프롬프트 수정, 버전 관리, A/B 테스트 가능
2. **YAML 구조**: `version`, `description`, `system`, `human`, `few_shots` 키
3. **load_prompt() 함수**: YAML → 메시지 리스트 → `ChatPromptTemplate` 변환
4. **Few-shot YAML**: `few_shots` 리스트의 `input`/`output` 쌍이 Human/Assistant 메시지로 변환됨

**다음 모듈**: `module_06_structured_output/01_pydantic_models.ipynb` - Pydantic으로 출력 스키마 정의